In [ ]:
pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=1a6ae127447070c023e3cc2d1729c699c2a28af72e7cda8d655785294e43958e
  Stored in directory: /root/.cache/pip/wheels/8f/ab/cb/45ccc40522d3a1c41e1d2ad53b8f33a62f394011ec38cd71c6
Successfully built wikipedia


In [ ]:
!pip install --upgrade langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.1 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.21
    Uninstalling langchain-0.3.21:
      Successfully uninstalled langchain-0.3.21


In [ ]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.0 MB/s eta 0:00:00


In [ ]:
import wikipedia
from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper

def get_wikipedia_summary(query):
    """Fetch summary from Wikipedia for a given query."""
    wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_tool.run(query)

def chat_with_bot(user_query):
    """Process user query and fetch relevant Wikipedia information."""
    wiki_summary = get_wikipedia_summary(user_query)
    response = f"Wikipedia Summary: {wiki_summary}"
    return response

if __name__ == "__main__":
    print("Chatbot: Ask me anything about Wikipedia topics!")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit", "bye"]:
            print("Chatbot: Goodbye!")
            break
        response = chat_with_bot(user_input)
        print(f"Chatbot: {response}")



Chatbot: Ask me anything about Wikipedia topics!
You: vijay


/usr/local/lib/python3.11/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.11/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Chatbot: Wikipedia Summary: Page: Vijay (actor)
Summary: Joseph Vijay Chandrasekhar (born 22 June 1974), known professionally as Vijay, is an Indian actor and playback singer who works in Tamil cinema. In a career spanning over three decades, Vijay has acted in 68 films and is one of the most commercially successful actors in Tamil cinema with multiple films amongst the highest-grossing Tamil films of all time and is amongst the highest paid actors in India. He has won several awards as an actor. Referred to as "Thalapathy" (transl. commander), Vijay has a significant fan following.
Born in Madras to director S. A. Chandrasekhar, Vijay made his debut as a child actor in the Tamil film Vetri (1984). After a few roles as a child actor in his father’s films, he played his first lead role in the film Naalaiya Theerpu (1992) at the age of 18. Vijay continued doing lead roles for the next few years with notable films amongst them included Poove Unakkaga (1996), Love Today (1997), Kadhalukku 

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 35.2 MB/s eta 0:00:00


In [ ]:
pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00


In [ ]:
pip install lxml


In [ ]:
# Wikipedia Chatbot using LangChain, OpenRouter API, and FAISS

import os
from langchain.chains import LLMChain
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.utilities import WikipediaAPIWrapper
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore import InMemoryDocstore
import faiss
from uuid import uuid4
from bs4 import BeautifulSoup

# Set OpenRouter API Key and Base URL
os.environ["OPENROUTER_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Initialize the language model and Wikipedia API wrapper
llm = OpenAI(temperature=0.7)
wikipedia = WikipediaAPIWrapper()

# Custom prompt template for generating answers based on Wikipedia content
prompt_template = PromptTemplate(
    input_variables=["query", "wiki_content"],
    template="""
You are a knowledgeable assistant. Use the following Wikipedia content to generate a well-formed, clear, and informative answer to the user's question. Avoid quoting directly:

Wikipedia Content: {wiki_content}
User Question: {query}
Answer:
"""
)

# Initialize FAISS for efficient search
embedding = OpenAIEmbeddings()
index = faiss.IndexFlatL2(1536)
docstore = InMemoryDocstore({})
index_to_docstore_id = {}
vector_store = FAISS(embedding, index, docstore, index_to_docstore_id)

# Create the LLM chain
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

print("Wikipedia Chatbot: Ask me anything! (Type 'exit' to quit)")

while True:
    user_input = input("You: ")
    if user_input.lower() == 'exit':
        print("Goodbye!")
        break

    try:
        # Fetch Wikipedia content
        wiki_content = wikipedia.run(user_input)
        if not wiki_content:
            print("Sorry, I couldn't find any information on that.")
            continue

        # Fixing BeautifulSoup warning
        soup = BeautifulSoup(wiki_content, features="lxml")
        cleaned_content = soup.get_text()

        # Split and embed the Wikipedia content
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        texts = text_splitter.split_text(cleaned_content)
        doc_ids = [str(uuid4()) for _ in texts]
        vector_store.add_texts(texts, doc_ids)

        # Search for relevant information
        search_results = vector_store.similarity_search(user_input, k=1)
        relevant_content = search_results[0]["text"] if search_results else "No relevant content found."

        # Generate a response using the LLM
        answer = llm_chain.run({"query": user_input, "wiki_content": relevant_content})

        print(f"Chatbot: {answer}")
    except Exception as e:
        print(f"An error occurred: {str(e)}")


Wikipedia Chatbot: Ask me anything! (Type 'exit' to quit)
You: dog


/usr/local/lib/python3.11/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.11/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


An error occurred: Error code: 404 - {'error': {'message': 'Not Found', 'code': 404}}
You: exit
Goodbye!


In [ ]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00


In [ ]:
# PDF-based QA Bot using LangChain and FAISS

import os
from langchain.chains import LLMChain
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore import InMemoryDocstore
import faiss
from uuid import uuid4
import PyPDF2
import wikipedia

# Set OpenAI API key and base URL for OpenRouter
os.environ["OPENROUTER_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Explicitly set the deployment name if necessary
# embeddings = OpenAIEmbeddings(deployment="YOUR_DEPLOYMENT_NAME", openai_api_base=os.environ.get("OPENAI_API_BASE"))
embeddings = OpenAIEmbeddings(openai_api_base=os.environ.get("OPENAI_API_BASE"))  # Use default deployment if not specified


# Initialize language model
llm = OpenAI(temperature=0.7, openai_api_base=os.environ.get("OPENAI_API_BASE")) # Use OpenRouter base URL for OpenAI calls

# ... (rest of the code remains the same)

# Initialize FAISS for efficient search
# embedding = OpenAIEmbeddings()  # This line is replaced with the above initialization
index = faiss.IndexFlatL2(1536)
docstore = InMemoryDocstore({})
index_to_docstore_id = {}
vector_store = FAISS(embeddings, index, docstore, index_to_docstore_id) # Use 'embeddings' variable here

# ... (rest of the code remains the same)

In [ ]:
pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.5 MB/s eta 0:00:00
